# Agent LLM de normalisation 

In [ ]:
# Cellule 1 — installation éventuelle
# À lancer seulement si les bibliothèques ne sont pas déjà installées.

# !pip install -U agno lxml pydantic
# !pip install ollama


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# Cellule 2 — imports

from pathlib import Path
from lxml import etree
from agno.agent import Agent
from agno.models.ollama import Ollama
from pydantic import BaseModel, Field
from typing import List, Literal
import json
import re

In [5]:
# Cellule 3 — configuration

DOSSIER_DATA = Path("data")

FICHIER_TEI = DOSSIER_DATA / "Frêne_volume_1.xml"
FICHIER_JSON_INTERMEDIAIRE = DOSSIER_DATA / "lignes_tei_intermediaire.json"
FICHIER_SORTIE = DOSSIER_DATA / "Frêne_volume_1_normalise.xml"
FICHIER_JOURNAL = DOSSIER_DATA / "journal_normalisations.json"

MODELE_OLLAMA = "qwen3:8b"

NS_TEI = "http://www.tei-c.org/ns/1.0"
NS = {"tei": NS_TEI}
XML_ID = "{http://www.w3.org/XML/1998/namespace}id"

In [6]:
# Cellule 4 — chargement TEI

def charger_tei(chemin_xml: Path) -> etree._ElementTree:
    """Charge un fichier XML-TEI."""
    parser = etree.XMLParser(remove_blank_text=False)
    return etree.parse(str(chemin_xml), parser)


arbre = charger_tei(FICHIER_TEI)
racine = arbre.getroot()

OSError: Error reading file 'data\Frêne_volume_1.xml': failed to load external entity "data/Frêne_volume_1.xml"

In [ ]:
# Cellule 5 — extraction JSON intermédiaire

def extraire_lignes_tei(arbre: etree._ElementTree) -> list[dict]:
    """
    Extrait les lignes TEI sous forme de JSON intermédiaire.
    
    Chaque entrée conserve :
    - xml_id : identifiant de la ligne ;
    - n : numéro de ligne ;
    - texte : contenu textuel.
    """
    lignes = arbre.xpath("//tei:sourceDoc//tei:line", namespaces=NS)

    resultats = []

    for i, ligne in enumerate(lignes):
        xml_id = ligne.get(XML_ID)

        if not xml_id:
            xml_id = f"ligne_sans_id_{i}"
            ligne.set(XML_ID, xml_id)

        texte = "".join(ligne.itertext()).strip()

        if texte:
            resultats.append({
                "xml_id": xml_id,
                "n": ligne.get("n"),
                "texte": texte
            })

    return resultats


lignes_json = extraire_lignes_tei(arbre)

with open(FICHIER_JSON_INTERMEDIAIRE, "w", encoding="utf-8") as f:
    json.dump(lignes_json, f, ensure_ascii=False, indent=2)

print(f"{len(lignes_json)} lignes exportées dans {FICHIER_JSON_INTERMEDIAIRE}")

In [ ]:
# Cellule 6 — schéma de réponse de l’agent

class PropositionNormalisation(BaseModel):
    """Normalisation proposée par l’agent."""

    forme_originale: str = Field(
        ...,
        description="Forme exacte observée dans le texte."
    )

    forme_normalisee: str = Field(
        ...,
        description="Forme normalisée proposée."
    )

    type: Literal["graphie", "abreviation", "toponyme"]


class ListeNormalisations(BaseModel):
    """Liste des normalisations proposées."""

    normalisations: List[PropositionNormalisation]

In [ ]:
# Cellule 7 — agent LLM local

agent_normalisation = Agent(
    model=Ollama(id=MODELE_OLLAMA),
    output_schema=ListeNormalisations,
    instructions=[
        "Tu es un assistant spécialisé en XML-TEI et transcriptions historiques françaises du XVIIIe siècle.",
        "Tu travailles sur un journal manuscrit jurassien.",
        "Ta tâche est de proposer une liste de normalisations utiles.",
        "Repère uniquement les formes effectivement présentes dans le JSON fourni.",
        "Cherche surtout les graphies anciennes, les abréviations et les toponymes.",
        "Exemples : etoit → était, cetoit → c’était, regnoit → régnait.",
        "Exemples : M: → Monsieur, Magd: → Madeleine, ds. → dans, ns → nous.",
        "Exemples : Bieñe → Bienne, Pery → Péry.",
        "Ne corrige pas librement l’OCR/HTR.",
        "Ne modernise pas tout le texte.",
        "Ne propose pas de correction incertaine.",
        "Retourne uniquement la structure demandée."
    ],
)

In [ ]:
# Cellule 8 — appel de l’agent sur le JSON intermédiaire

def generer_normalisations_depuis_json(
    lignes_json: list[dict],
    limite_lignes: int = 300
) -> dict:
    """
    Envoie un échantillon du JSON intermédiaire à l’agent.
    
    limite_lignes évite de surcharger le modèle local.
    """
    echantillon = lignes_json[:limite_lignes]

    prompt = f"""
Voici un JSON extrait d’un XML-TEI.

Objectif :
proposer une liste globale de normalisations TEI.

Contraintes :
- forme_originale doit apparaître exactement dans le texte ;
- ne proposer que graphie, abreviation ou toponyme ;
- ne pas corriger librement l’OCR/HTR ;
- ne pas moderniser stylistiquement le texte.

JSON :
{json.dumps(echantillon, ensure_ascii=False, indent=2)}
"""

    reponse = agent_normalisation.run(prompt)
    contenu = reponse.content

    if isinstance(contenu, ListeNormalisations):
        propositions = contenu.normalisations
    elif isinstance(contenu, dict):
        propositions = [
            PropositionNormalisation(**p)
            for p in contenu.get("normalisations", [])
        ]
    else:
        raise ValueError("Réponse LLM non exploitable.")

    corpus_complet = "\n".join(ligne["texte"] for ligne in lignes_json)

    normalisations = {}

    for prop in propositions:
        forme_originale = prop.forme_originale.strip()
        forme_normalisee = prop.forme_normalisee.strip()

        if not forme_originale:
            continue

        if forme_originale not in corpus_complet:
            continue

        identifiant = nettoyer_id_xml(forme_originale)

        normalisations[forme_originale] = {
            "forme_normalisee": forme_normalisee,
            "type": prop.type,
            "id": identifiant,
            "balise_originale": "abbr" if prop.type == "abreviation" else "orig",
            "balise_normalisee": "expan" if prop.type == "abreviation" else "reg",
        }

    return normalisations

In [ ]:
# Cellule 9 — fonction utilitaire pour les xml:id

def nettoyer_id_xml(texte: str) -> str:
    """Transforme une forme en identifiant XML utilisable."""
    texte = texte.lower()
    texte = texte.replace(":", "")
    texte = texte.replace(".", "")
    texte = texte.replace(" ", "_")
    texte = texte.replace("’", "")
    texte = texte.replace("'", "")
    texte = re.sub(r"[^a-zA-Z0-9_\-éèêëàâîïôöûüçñ]", "", texte)

    if not texte:
        texte = "forme"

    return f"norm_{texte}"

In [ ]:
# Cellule 10 — génération de la liste de normalisations

NORMALISATIONS = generer_normalisations_depuis_json(lignes_json)

print(f"{len(NORMALISATIONS)} normalisations proposées :")

for forme, infos in NORMALISATIONS.items():
    print(forme, "→", infos["forme_normalisee"], f"({infos['type']})")

In [ ]:
# Cellule 11 — correction manuelle facultative

# Tu peux ajouter, corriger ou supprimer ici avant application.

# Exemple :
# NORMALISATIONS["etoit"] = {
#     "forme_normalisee": "était",
#     "type": "graphie",
#     "id": "norm_etoit",
#     "balise_originale": "orig",
#     "balise_normalisee": "reg"
# }

# Exemple :
# del NORMALISATIONS["forme_non_souhaitee"]

In [ ]:
# Cellule 12 — création d’éléments TEI

def tei_element(nom: str, texte: str | None = None, **attributs):
    """Crée un élément TEI."""
    element = etree.Element(f"{{{NS_TEI}}}{nom}", **attributs)
    element.text = texte
    return element


def creer_choice(forme_originale: str, infos: dict) -> etree._Element:
    """Crée un <choice corresp='#...'>."""
    choice = tei_element("choice", corresp=f"#{infos['id']}")

    original = etree.SubElement(
        choice,
        f"{{{NS_TEI}}}{infos['balise_originale']}"
    )
    original.text = forme_originale

    normalise = etree.SubElement(
        choice,
        f"{{{NS_TEI}}}{infos['balise_normalisee']}"
    )
    normalise.text = infos["forme_normalisee"]

    return choice

In [ ]:
# Cellule 13 — repérage dans les fragments de texte

def trouver_normalisations(texte: str) -> list[dict]:
    """Repère les occurrences normalisables dans un texte."""
    occurrences = []

    for forme_originale, infos in NORMALISATIONS.items():
        position = 0

        while True:
            index = texte.find(forme_originale, position)

            if index == -1:
                break

            occurrences.append({
                "forme_originale": forme_originale,
                "debut": index,
                "fin": index + len(forme_originale),
                "infos": infos
            })

            position = index + len(forme_originale)

    occurrences.sort(
        key=lambda x: (
            x["debut"],
            -(x["fin"] - x["debut"])
        )
    )

    return occurrences


def filtrer_chevauchements(occurrences: list[dict]) -> list[dict]:
    """Supprime les occurrences qui se chevauchent."""
    resultat = []
    derniere_fin = -1

    for occurrence in occurrences:
        if occurrence["debut"] >= derniere_fin:
            resultat.append(occurrence)
            derniere_fin = occurrence["fin"]

    return resultat

In [ ]:
# Cellule 14 — transformation d’un fragment texte en <choice>

def fragmenter_texte_avec_choice(texte: str):
    """
    Transforme un fragment textuel en texte + nœuds <choice>.
    
    Retour :
    - texte initial ;
    - nouveaux nœuds ;
    - journal.
    """
    occurrences = filtrer_chevauchements(
        trouver_normalisations(texte)
    )

    if not occurrences:
        return texte, [], []

    texte_initial = ""
    nouveaux_noeuds = []
    journal = []

    position = 0
    dernier = None

    for occurrence in occurrences:
        avant = texte[position:occurrence["debut"]]

        if dernier is None:
            texte_initial += avant
        else:
            dernier.tail = (dernier.tail or "") + avant

        choice = creer_choice(
            occurrence["forme_originale"],
            occurrence["infos"]
        )

        nouveaux_noeuds.append(choice)
        dernier = choice

        position = occurrence["fin"]

        journal.append({
            "forme_originale": occurrence["forme_originale"],
            "forme_normalisee": occurrence["infos"]["forme_normalisee"],
            "type": occurrence["infos"]["type"],
            "id_normalisation": occurrence["infos"]["id"]
        })

    reste = texte[position:]

    if dernier is None:
        texte_initial += reste
    else:
        dernier.tail = (dernier.tail or "") + reste

    return texte_initial, nouveaux_noeuds, journal

In [ ]:
# Cellule 15 — normalisation sans détruire les balises internes

BALISES_A_NE_PAS_NORMALISER = {
    f"{{{NS_TEI}}}choice",
    f"{{{NS_TEI}}}orig",
    f"{{{NS_TEI}}}reg",
    f"{{{NS_TEI}}}abbr",
    f"{{{NS_TEI}}}expan",
}


def inserer_noeuds_apres(parent: etree._Element, index: int, noeuds: list):
    """Insère des nœuds dans un parent XML."""
    for decalage, noeud in enumerate(noeuds):
        parent.insert(index + decalage, noeud)


def normaliser_text_direct(element: etree._Element) -> list[dict]:
    """Normalise element.text sans toucher aux enfants XML."""
    texte = element.text

    if not texte:
        return []

    nouveau_texte, noeuds, journal = fragmenter_texte_avec_choice(texte)

    if not noeuds:
        return []

    element.text = nouveau_texte
    inserer_noeuds_apres(element, 0, noeuds)

    return journal


def normaliser_tail(enfant: etree._Element) -> list[dict]:
    """Normalise le texte situé après un enfant XML."""
    texte = enfant.tail

    if not texte:
        return []

    nouveau_tail, noeuds, journal = fragmenter_texte_avec_choice(texte)

    if not noeuds:
        return []

    parent = enfant.getparent()

    if parent is None:
        return []

    index_enfant = parent.index(enfant)

    enfant.tail = nouveau_tail
    inserer_noeuds_apres(parent, index_enfant + 1, noeuds)

    return journal


def normaliser_element_recursif(
    element: etree._Element,
    contexte_ligne: etree._Element
) -> list[dict]:
    """Normalise récursivement les textes et tails d’une ligne."""
    journal = []

    if element.tag in BALISES_A_NE_PAS_NORMALISER:
        return journal

    changements = normaliser_text_direct(element)

    for changement in changements:
        changement["ligne_xml_id"] = contexte_ligne.get(XML_ID)
        changement["ligne_n"] = contexte_ligne.get("n")

    journal.extend(changements)

    for enfant in list(element):
        if enfant.tag not in BALISES_A_NE_PAS_NORMALISER:
            journal.extend(
                normaliser_element_recursif(enfant, contexte_ligne)
            )

        changements_tail = normaliser_tail(enfant)

        for changement in changements_tail:
            changement["ligne_xml_id"] = contexte_ligne.get(XML_ID)
            changement["ligne_n"] = contexte_ligne.get("n")

        journal.extend(changements_tail)

    return journal

In [ ]:
# Cellule 16 — réinjection dans le TEI grâce aux xml:id

def indexer_lignes_par_id(arbre: etree._ElementTree) -> dict:
    """Crée un index xml_id → élément <line>."""
    lignes = arbre.xpath("//tei:sourceDoc//tei:line", namespaces=NS)

    return {
        ligne.get(XML_ID): ligne
        for ligne in lignes
        if ligne.get(XML_ID)
    }


def reinjecter_normalisations(arbre: etree._ElementTree, lignes_json: list[dict]):
    """
    Réinjecte les normalisations dans les lignes du TEI original
    à partir des identifiants du JSON intermédiaire.
    """
    index_lignes = indexer_lignes_par_id(arbre)

    journal = []

    for entree in lignes_json:
        xml_id = entree["xml_id"]

        ligne = index_lignes.get(xml_id)

        if ligne is None:
            continue

        changements = normaliser_element_recursif(ligne, ligne)
        journal.extend(changements)

    return journal

In [ ]:
# Cellule 17 — création du <standOff>

def supprimer_standoff_existant(racine: etree._Element):
    """Supprime un ancien <standOff type='normalisation'>."""
    standoffs = racine.xpath(
        "./tei:standOff[@type='normalisation']",
        namespaces=NS
    )

    for standoff in standoffs:
        racine.remove(standoff)


def creer_standoff_normalisations(ids_utilises: set[str]) -> etree._Element:
    """Crée le <standOff> global des normalisations utilisées."""
    standoff = tei_element("standOff", type="normalisation")

    liste = etree.SubElement(
        standoff,
        f"{{{NS_TEI}}}list",
        type="normalisation",
        attrib={XML_ID: "normalisations"}
    )

    for forme_originale, infos in NORMALISATIONS.items():
        if infos["id"] not in ids_utilises:
            continue

        item = etree.SubElement(
            liste,
            f"{{{NS_TEI}}}item",
            type=infos["type"],
            attrib={XML_ID: infos["id"]}
        )

        original = etree.SubElement(
            item,
            f"{{{NS_TEI}}}{infos['balise_originale']}"
        )
        original.text = forme_originale

        normalise = etree.SubElement(
            item,
            f"{{{NS_TEI}}}{infos['balise_normalisee']}"
        )
        normalise.text = infos["forme_normalisee"]

    return standoff

In [ ]:
# Cellule 18 — déclaration éditoriale

def ajouter_declaration_normalisation(arbre: etree._ElementTree):
    """Ajoute une déclaration de normalisation dans le teiHeader."""
    encoding_desc = arbre.xpath(
        "//tei:teiHeader/tei:encodingDesc",
        namespaces=NS
    )

    if not encoding_desc:
        return

    encoding_desc = encoding_desc[0]

    editorial_decl = encoding_desc.find(f"{{{NS_TEI}}}editorialDecl")

    if editorial_decl is None:
        editorial_decl = etree.SubElement(
            encoding_desc,
            f"{{{NS_TEI}}}editorialDecl"
        )

    ancienne = editorial_decl.find(f"{{{NS_TEI}}}normalization")

    if ancienne is not None:
        editorial_decl.remove(ancienne)

    normalization = etree.SubElement(
        editorial_decl,
        f"{{{NS_TEI}}}normalization"
    )

    p = etree.SubElement(normalization, f"{{{NS_TEI}}}p")
    p.text = (
        "Les normalisations graphiques, abréviatives et toponymiques "
        "sont encodées localement avec l’élément choice. "
        "Chaque choice renvoie à une entrée globale du standOff "
        "au moyen de l’attribut corresp. "
        "La liste des normalisations a été proposée par un agent LLM local "
        "à partir d’un JSON intermédiaire extrait du TEI."
    )

In [ ]:
# Cellule 19 — exécution complète

journal = reinjecter_normalisations(arbre, lignes_json)

ids_utilises = {
    entree["id_normalisation"]
    for entree in journal
}

supprimer_standoff_existant(racine)

standoff = creer_standoff_normalisations(ids_utilises)
racine.append(standoff)

ajouter_declaration_normalisation(arbre)

print(f"{len(lignes_json)} lignes traitées depuis le JSON intermédiaire.")
print(f"{len(journal)} normalisations appliquées.")
print(f"{len(ids_utilises)} règles déclarées dans le standOff.")

In [ ]:
# Cellule 20 — sauvegardes

arbre.write(
    str(FICHIER_SORTIE),
    encoding="UTF-8",
    xml_declaration=True,
    pretty_print=True
)

with open(FICHIER_JOURNAL, "w", encoding="utf-8") as f:
    json.dump(journal, f, ensure_ascii=False, indent=2)

print(f"TEI enrichi sauvegardé : {FICHIER_SORTIE}")
print(f"Journal sauvegardé : {FICHIER_JOURNAL}")

In [ ]:
# Cellule 21 — contrôle rapide

for entree in journal[:30]:
    print(
        entree["ligne_n"],
        entree["forme_originale"],
        "→",
        entree["forme_normalisee"],
        f"({entree['type']})"
    )